# Sector coupling et gestion du carbone — TD corrigé

**Public visé :** élèves ingénieurs de 5e année ayant déjà étudié les bases de l’optimisation linéaire et les fondamentaux de PyPSA.

Ce TD suit le TD d’optimisation à court terme et précède celui consacré à la planification des capacités. Il explore trois applications :
1. **Modèle à un nœud avec couplage sectoriel** : électricité, hydrogène, chaleur et mobilité électrique ;
2. **Production de méthanol en système isolé** : dimensionnement conjoint des renouvelables, des convertisseurs et des stockages ;
3. **Gestion des flux de carbone** : biomasse, gaz, captage, utilisation du CO₂ et objectif d’émissions nettes négatives.

Les explications sont en français. Les blocs de code de correction sont délimités par `### DEBUT SOLUTION` et `### FIN SOLUTION` afin de permettre la génération automatique d’une version énoncé.

**Dépendances :** `pypsa`, `pandas`, `numpy`, `matplotlib`. Plusieurs séries temporelles et un réseau de départ sont téléchargés depuis des ressources externes : une connexion Internet est nécessaire. Les API et données peuvent évoluer selon les versions de PyPSA et des sources.

> **Conventions de ce notebook**
> - Les cellules `### DEBUT SOLUTION … ### FIN SOLUTION` contiennent le code à écrire ; les fonctions « fournies » (hors balises) restent dans l'énoncé.
> - Les cellules **« Éléments de réponse »** portent la balise de cellule `correction` : à retirer pour générer l'énoncé.
> - Chaque exercice se termine par une section **« Pour explorer »** : on fait varier un paramètre pour mettre en évidence un phénomène classique.
> - 💡 **Prédisez** le résultat *avant* d'exécuter la cellule ; copiez toujours un réseau (`copier_reseau(n)`) avant de le modifier.


## Préparation commune

In [ ]:
### DEBUT SOLUTION
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa

warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

print("Version de PyPSA :", pypsa.__version__)
### FIN SOLUTION


In [ ]:
def copier_reseau(n):
    """Copie un réseau, y compris s'il a déjà été optimisé.

    Dans les versions récentes de PyPSA, `n.copy()` refuse de copier un réseau résolu
    tant que le modèle du solveur est attaché : on le détache d'abord.
    """
    if getattr(n, "model", None) is not None:
        n.model.solver_model = None
    return n.copy()


logging.getLogger("pypsa").setLevel(logging.ERROR)     # messages d'information masqués (on relance beaucoup de calculs)
logging.getLogger("linopy").setLevel(logging.WARNING)  # on garde les alertes, par exemple en cas d'infaisabilité

---

# Exercice 1 — Modèle à un nœud avec couplage sectoriel

## Objectifs

On part d’un modèle électrique existant à un seul nœud, puis on ajoute progressivement de nouveaux vecteurs énergétiques :
- l’hydrogène, produit par électrolyse et éventuellement reconverti en électricité ;
- la chaleur, fournie par une pompe à chaleur ou une chaudière électrique résistive ;
- la cogénération, qui produit simultanément électricité et chaleur ;
- les véhicules électriques, avec recharge flexible et possibilité de réinjection vers le réseau.

Le **couplage sectoriel** consiste à représenter ces interactions dans un même problème d’optimisation. Les technologies et les demandes ne sont plus étudiées séparément : elles se concurrencent pour les ressources et peuvent se compléter dans le temps.


## 1.1 Charger le modèle électrique de départ

Le réseau téléchargé contient un modèle de planification des capacités électriques pour l’Allemagne, avec une résolution temporelle de quatre heures. Il comprend notamment des capacités renouvelables, une centrale au gaz et du stockage.

In [ ]:
### DEBUT SOLUTION
url = "https://tubcloud.tu-berlin.de/s/pzytNg9gtkgPpXc/download/network-cem.nc"
n = pypsa.Network(url)
display(n)
### FIN SOLUTION


## 1.2 Représenter séparément la chaîne hydrogène

On remplace le stockage hydrogène simplifié par trois composants explicites : un bus hydrogène, un électrolyseur, une pile à combustible et un stockage. Cela permet de distinguer les puissances des convertisseurs de la capacité énergétique du réservoir.

In [ ]:
### DEBUT SOLUTION
n.remove("StorageUnit", "hydrogen storage underground")
n.add("Bus", "hydrogen", carrier="hydrogen")

n.add(
    "Link", "electrolysis",
    bus0="electricity", bus1="hydrogen",
    carrier="electrolysis", p_nom_extendable=True,
    efficiency=0.7, capital_cost=50e3,
)
n.add(
    "Link", "fuel cell",
    bus0="hydrogen", bus1="electricity",
    carrier="fuel cell", p_nom_extendable=True,
    efficiency=0.5, capital_cost=120e3,
)
n.add(
    "Store", "hydrogen storage",
    bus="hydrogen", carrier="hydrogen storage",
    capital_cost=140, e_nom_extendable=True, e_cyclic=True,
)

# Demande d'hydrogène constante, égale à la demande électrique moyenne.
p_h2 = n.loads_t.p_set["demand"].mean()
n.add("Load", "hydrogen demand", bus="hydrogen", carrier="hydrogen", p_set=p_h2)
### FIN SOLUTION


**À observer.** L’électrolyseur consomme de l’électricité pour produire de l’hydrogène ; la pile à combustible réalise la conversion inverse avec des pertes. Le stockage permet de décaler la production et la consommation dans le temps.

## 1.3 Ajouter la demande de chaleur et une pompe à chaleur

Le coefficient de performance (*COP*) d’une pompe à chaleur dépend de la différence entre la température de sortie et celle de la source. Un COP de 3 signifie qu’un MWh d’électricité permet de fournir 3 MWh de chaleur dans le modèle.

In [ ]:
### DEBUT SOLUTION
n.add("Bus", "heat", carrier="heat")

url = "https://tubcloud.tu-berlin.de/s/mSkHERH8fJCKNXx/download/heat-load-example.csv"
heat_demand = pd.read_csv(url, index_col=0, parse_dates=True).squeeze()
n.add("Load", "heat demand", carrier="heat", bus="heat", p_set=heat_demand)

def cop(t_source, t_sink=55):
    delta_t = t_sink - t_source
    return 6.81 - 0.121 * delta_t + 0.000630 * delta_t**2

url = "https://tubcloud.tu-berlin.de/s/S4jRAQMP5Te96jW/download/ninja_weather_country_DE_merra-2_population_weighted.csv"
temp = pd.read_csv(url, skiprows=2, index_col=0, parse_dates=True).loc["2015", "temperature"][::4]
cop_profile = cop(temp)

n.add(
    "Link", "heat pump",
    carrier="heat pump", bus0="electricity", bus1="heat",
    efficiency=cop_profile, p_nom_extendable=True, capital_cost=3e5,
)
n.add(
    "Link", "resistive heater",
    carrier="resistive heater", bus0="electricity", bus1="heat",
    efficiency=0.9, p_nom_extendable=True, capital_cost=1e4,
)
### FIN SOLUTION


La pompe à chaleur est généralement plus efficace qu’un chauffage résistif, mais elle nécessite un investissement différent et son COP varie avec la température. Le chauffage résistif peut rester utile comme technologie d’appoint.

## 1.4 Ajouter du gaz et une cogénération

La cogénération (CHP) produit simultanément de l’électricité et de la chaleur à partir d’un même flux de combustible. Elle est représentée par un `Link` multi-sorties : `bus1` reçoit l’électricité et `bus2` reçoit la chaleur. Dans cet exemple pédagogique, on utilise du méthane vert à coût marginal élevé.

In [ ]:
### DEBUT SOLUTION
# La contrainte CO2 du modèle électrique initial n'est plus utilisée dans cette variante.
if "CO2Limit" in n.global_constraints.index:
    n.remove("GlobalConstraint", "CO2Limit")

n.add("Bus", "gas", carrier="gas")
n.add(
    "Store", "gas storage",
    carrier="gas storage", e_initial=100e6, e_nom=100e6,
    bus="gas", marginal_cost=150,
)
n.remove("Generator", "OCGT")
n.add(
    "Link", "OCGT",
    bus0="gas", bus1="electricity", carrier="OCGT",
    p_nom_extendable=True, capital_cost=20_000, efficiency=0.4,
)
n.add(
    "Link", "CHP",
    bus0="gas", bus1="electricity", bus2="heat",
    carrier="CHP", p_nom_extendable=True, capital_cost=40_000,
    efficiency=0.4, efficiency2=0.4,
)
### FIN SOLUTION


## 1.5 Ajouter les véhicules électriques

On ajoute un bus dédié aux véhicules électriques, leur profil de demande et une disponibilité de recharge à domicile. La puissance maximale du chargeur est obtenue à partir du nombre de véhicules et de la puissance unitaire du chargeur. Le lien V2G permet de réinjecter une partie de l’énergie vers le réseau lorsque les véhicules sont branchés.

In [ ]:
### DEBUT SOLUTION
n.add("Bus", "EV", carrier="EV")

url = "https://tubcloud.tu-berlin.de/s/9r5bMSbzzQiqG7H/download/electric-vehicle-profile-example.csv"
ev_demand = pd.read_csv(url, index_col=0, parse_dates=True).squeeze()
n.add("Load", "EV demand", bus="EV", carrier="EV demand", p_set=ev_demand)

url = "https://tubcloud.tu-berlin.de/s/E3PBWPfYaWwCq7a/download/electric-vehicle-availability-example.csv"
availability_profile = pd.read_csv(url, index_col=0, parse_dates=True).squeeze()

number_cars = 40e6
charger_rate = 0.011  # MW par véhicule, soit 11 kW
p_nom_ev = number_cars * charger_rate

n.add(
    "Link", "EV charger",
    bus0="electricity", bus1="EV", carrier="EV charger",
    p_nom=p_nom_ev, p_max_pu=availability_profile, efficiency=0.9,
)
n.add(
    "Link", "V2G",
    bus0="EV", bus1="electricity", carrier="V2G",
    p_nom=p_nom_ev, p_max_pu=availability_profile, efficiency=0.9,
)
### FIN SOLUTION


## 1.6 Représenter la recharge intelligente

Le potentiel de recharge intelligente est modélisé par un `Store` dont l’état de charge doit respecter un profil minimal. Ce profil représente une exigence de disponibilité de l’énergie dans les batteries à certains moments, par exemple le matin.

In [ ]:
### DEBUT SOLUTION
bev_energy = 0.05  # MWh par véhicule
dsm_participants = 0.5
e_nom_ev = number_cars * bev_energy * dsm_participants

url = "https://tubcloud.tu-berlin.de/s/K62yACBRTrxLTia/download/dsm-profile-example.csv"
dsm_profile = pd.read_csv(url, index_col=0, parse_dates=True).squeeze().shift(2, fill_value=0)

n.add(
    "Store", "EV DSM",
    bus="EV", carrier="EV battery", e_cyclic=True,
    e_nom=e_nom_ev, e_min_pu=dsm_profile.loc[n.snapshots],
)
### FIN SOLUTION


## 1.7 Résoudre et analyser le système couplé

L’optimisation choisit les capacités extensibles et leur fonctionnement temporel en tenant compte simultanément de tous les secteurs. Les résultats permettent de comparer les coûts d’investissement et les flux énergétiques.

In [ ]:
### DEBUT SOLUTION
n.optimize()
display(n.statistics())
### FIN SOLUTION


In [ ]:
### DEBUT SOLUTION
n.statistics.capex().div(1e9).sort_values().dropna().plot.bar(
    ylabel="Investissement annualisé (milliards €/an)", figsize=(9, 4)
)
plt.show()

n.loads_t.p_set.div(1e3).plot(figsize=(10, 3), ylabel="GW", title="Profils de demande")
plt.show()

n.statistics.energy_balance.plot.area(linewidth=0, bus_carrier="electricity")
plt.title("Bilan énergétique — électricité")
plt.show()

n.statistics.energy_balance.plot.area(linewidth=0, bus_carrier="heat")
plt.title("Bilan énergétique — chaleur")
plt.show()
### FIN SOLUTION


## 1.8 Pour explorer : chaleur, prix couplés et valeur des options de flexibilité

> ⏱️ La résolution du modèle complet peut prendre de une à quelques minutes ; lancez les variantes de 1.8.c une par une si besoin.

### 1.8.a Qui fournit la chaleur, et quand ?

Trois technologies produisent de la chaleur : la **pompe à chaleur** (rendement = COP, qui dépend de la température extérieure), le **chauffage résistif** (rendement 0,9) et la **cogénération**. Le COP varie beaucoup dans l'année : commencez par le visualiser, puis regardez la production hebdomadaire de chaque technologie.

> 💡 **Prédisez** : en hiver, quand il fait très froid, quelle technologie est la plus sollicitée en *proportion* ? Pourquoi le chauffage résistif existe-t-il dans le mix si la pompe à chaleur est plus efficace ?

In [ ]:
# TODO : (1) tracer le COP en fonction de la température et en donner la plage de variation ;
# (2) tracer la puissance thermique moyenne hebdomadaire de chaque technologie (GW).
### DEBUT SOLUTION
cop_pac = n.links_t.efficiency["heat pump"]
print(f"COP : min {cop_pac.min():.2f}, moyenne {cop_pac.mean():.2f}, max {cop_pac.max():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].scatter(temp.loc[n.snapshots], cop_pac, s=3)
axes[0].set_xlabel("Température extérieure (°C)")
axes[0].set_ylabel("COP de la pompe à chaleur")

chaleur = pd.DataFrame({
    "pompe à chaleur": -n.links_t.p1["heat pump"],
    "chauffage résistif": -n.links_t.p1["resistive heater"],
    "cogénération": -n.links_t.p2["CHP"],
})
chaleur.resample("W").mean().div(1e3).plot.area(ax=axes[1], linewidth=0, ylabel="GW thermiques (moyenne hebdo)")
plt.tight_layout()
plt.show()

display((chaleur.mul(n.snapshot_weightings.generators, axis=0).sum() / 1e6).round(2).rename("Chaleur produite (TWh)"))
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

- Le COP **diminue quand il fait froid** (plus l'écart entre la source et la température de sortie est grand, plus le rendement baisse). Il est donc le plus faible précisément au moment où le besoin de chaleur est maximal.
- La pompe à chaleur assure l'essentiel de l'énergie (technologie de « base » : coût d'investissement élevé, énergie très efficace). Le **chauffage résistif** a un coût d'investissement très faible mais un rendement faible : c'est une technologie de **pointe**, utilisée surtout lors des vagues de froid, où il évite de surdimensionner la pompe à chaleur. On retrouve la logique des courbes de dépistage du TD long terme.
- La **cogénération** produit de la chaleur en même temps que de l'électricité quand le système a besoin des deux.

### 1.8.b Le prix de la chaleur est lié au prix de l'électricité

Le couplage sectoriel **connecte les prix** : si la pompe à chaleur est la technologie marginale (en fonctionnement sans être à sa capacité maximale), le prix de la chaleur vaut celui de l'électricité divisé par le COP : $\lambda_{\text{chaleur}} = \lambda_{\text{élec}} / \text{COP}$. On vérifie cette relation et on la représente.

In [ ]:
# TODO : repérer les instants où la pompe à chaleur fonctionne sans être saturée, calculer l'écart à la relation ci-dessus,
# puis tracer le prix de la chaleur en fonction de prix_élec / COP.
### DEBUT SOLUTION
prix = n.buses_t.marginal_price[["electricity", "heat"]]
p_pac = n.links_t.p0["heat pump"]
cap_pac = n.links.loc["heat pump", "p_nom_opt"]
libre = (p_pac > 1e-3 * cap_pac) & (p_pac < 0.999 * cap_pac)

ecart = (prix["heat"] - prix["electricity"] / cop_pac)[libre]
print(f"{libre.sum()} instants avec pompe à chaleur non saturée ; écart maximal : {ecart.abs().max():.4f} €/MWh")

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.scatter(prix["electricity"] / cop_pac, prix["heat"], s=3, color="lightgrey", label="autres instants")
ax.scatter((prix["electricity"] / cop_pac)[libre], prix["heat"][libre], s=4, label="PAC non saturée")
lim = prix["heat"].max()
ax.plot([0, lim], [0, lim], "k--", lw=0.8)
ax.set_xlabel("Prix de l'électricité / COP (€/MWh)")
ax.set_ylabel("Prix de la chaleur (€/MWh)")
ax.legend()
plt.show()
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

Quand la pompe à chaleur est marginale, le prix de la chaleur est **exactement** celui de l'électricité divisé par le COP (écart numérique nul) : les points sont sur la droite $y=x$. Hors de ces instants, la chaleur est plus chère que cette valeur : la pompe à chaleur est **saturée** (dernier MW de capacité utilisé), c'est alors le chauffage résistif ou la cogénération qui fixe le prix de la chaleur. Ainsi, **un choc sur le prix de l'électricité se répercute sur la chaleur** (et inversement, via la cogénération) : c'est tout l'intérêt de modéliser les secteurs ensemble.

### 1.8.c Que vaut chaque option de flexibilité ?

On retire une option à la fois (pompe à chaleur, cogénération, V2G) et on mesure l'augmentation du coût total du système (investissement annualisé + exploitation). Retirer une option ne peut jamais réduire le coût : l'écart mesure la **valeur de l'option** pour le système.

> 💡 **Prédisez** le classement des trois options par valeur décroissante.

In [ ]:
def resoudre_sans(n_ref, retirer=()):
    """Résout une variante du réseau sans certains composants, ex. retirer=[("Link", "V2G")]."""
    m = copier_reseau(n_ref)
    for composant, nom in retirer:
        m.remove(composant, nom)
    statut, condition = m.optimize()
    assert statut == "ok", f"Optimisation non aboutie : {condition}"
    return m


def cout_total(m):
    return (m.statistics.capex().sum() + m.statistics.opex().sum()) / 1e9   # milliards d'euros par an

In [ ]:
# TODO : résoudre les trois variantes et construire un tableau (coût total en G€/an, surcoût par rapport à la référence,
# capacités en GW des liens de chaleur et de cogénération).
### DEBUT SOLUTION
variantes = {
    "référence": n,
    "sans pompe à chaleur": resoudre_sans(n, [("Link", "heat pump")]),
    "sans cogénération": resoudre_sans(n, [("Link", "CHP")]),
    "sans V2G": resoudre_sans(n, [("Link", "V2G")]),
}
lignes = {}
for nom, m in variantes.items():
    cap = m.links.groupby("carrier").p_nom_opt.sum() / 1e3
    lignes[nom] = {
        "Coût total (G€/an)": cout_total(m),
        "PAC (GW)": cap.get("heat pump", 0),
        "Chauffage résistif (GW)": cap.get("resistive heater", 0),
        "CHP (GW)": cap.get("CHP", 0),
    }
tableau = pd.DataFrame(lignes).T
tableau["Surcoût (G€/an)"] = tableau["Coût total (G€/an)"] - tableau.loc["référence", "Coût total (G€/an)"]
display(tableau.round(2))
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

Chaque variante a un coût **supérieur ou égal** à la référence. Le surcoût mesure la valeur de l'option retirée ; les capacités des technologies restantes augmentent pour compenser (par exemple plus de chauffage résistif sans pompe à chaleur). Comparez les trois surcoûts : l'option dont la suppression coûte le plus est celle dont le système dépend le plus. Une option qui coûte très peu à supprimer est **substituable** (d'autres technologies remplissent la même fonction à peu de frais). Le V2G, en particulier, a une valeur limitée si d'autres flexibilités bon marché existent déjà.

**Questions de synthèse**
1. Pourquoi la valeur d'une option de flexibilité dépend-elle des autres options présentes dans le système ?
2. Dans quels cas la pompe à chaleur *augmente-t-elle* les besoins de flexibilité électrique plutôt que de les réduire ?

**Questions de discussion**
- Quel est l’intérêt de représenter explicitement l’hydrogène plutôt qu’un stockage électrique équivalent ?
- Comment la température extérieure modifie-t-elle la valeur de la pompe à chaleur ?
- Quel compromis la cogénération impose-t-elle entre production électrique et production de chaleur ?
- Dans quelles conditions le V2G et la recharge intelligente peuvent-ils réduire les besoins de flexibilité du système ?


---

# Exercice 2 — Production de méthanol en système isolé

## Objectifs

Dimensionner un système autonome qui produit du méthanol à partir d’électricité renouvelable, d’hydrogène et de CO₂. Le modèle choisit les capacités de production éolienne et solaire, des batteries, de l’électrolyse, de la turbine, des réservoirs et de l’unité de méthanolisation.

Le système est dit **isolé** car il n’échange pas d’électricité avec un réseau externe. Il doit satisfaire une demande annuelle constante de méthanol. Les coûts et rendements sont issus du jeu de données technico-économiques PyPSA pour 2030, et les profils renouvelables proviennent de model.energy.


## 2.1 Charger les hypothèses technico-économiques

Les coûts d’investissement sont annualisés à l’aide du facteur d’annuité. Pour les technologies exprimées en €/kW, on convertit les valeurs en €/MW. On utilise ici les données du dépôt `PyPSA/technology-data`.

In [ ]:
### DEBUT SOLUTION
from pypsa.costs import annuity

YEAR = 2030
url = f"https://raw.githubusercontent.com/PyPSA/technology-data/master/outputs/costs_{YEAR}.csv"
costs = pd.read_csv(url, index_col=[0, 1])
costs.loc[costs.unit.str.contains("/kW"), "value"] *= 1e3
costs = costs.value.unstack().fillna({"discount rate": 0.07, "lifetime": 20, "FOM": 0})
costs.loc["electrolysis", "investment"] = 500

annuity_factor = costs.apply(
    lambda row: annuity(row["discount rate"], row["lifetime"]), axis=1
)
costs["capital_cost"] = (
    annuity_factor + costs["FOM"] / 100
) * costs["investment"]
### FIN SOLUTION


## 2.2 Charger les profils éolien et solaire

On utilise une résolution de quatre heures pour réduire la taille du problème. Le lien ci-dessous correspond au sud de l’Argentine ; une autre série temporelle, pour la Namibie, est indiquée en commentaire dans le code.

In [ ]:
### DEBUT SOLUTION
RESOLUTION = 4  # heures
url = "https://model.energy/data/time-series-ca2bcb9e843aeb286cd6295854c885b6.csv"
# Variante Namibie :
# url = "https://model.energy/data/time-series-57f7bbcb5c4821506de052e52d022b48.csv"
ts = pd.read_csv(url, index_col=0, parse_dates=True)[::RESOLUTION]
display(ts.head())
### FIN SOLUTION


## 2.3 Construire les bus et la demande de méthanol

Chaque vecteur énergétique possède son propre bus. La demande est constante et correspond à une production annuelle cible de 1 TWh de méthanol. Les snapshots sont pondérés par quatre heures pour tenir compte de la résolution temporelle.

In [ ]:
### DEBUT SOLUTION
n_methanol = pypsa.Network()
for carrier in ["electricity", "hydrogen", "co2", "methanol"]:
    unit = "t/h" if carrier == "co2" else "MW"
    n_methanol.add("Bus", carrier, carrier=carrier, unit=unit)

n_methanol.set_snapshots(ts.index)
n_methanol.snapshot_weightings.loc[:, :] = RESOLUTION

carriers = {
    "wind": "dodgerblue", "solar": "gold",
    "hydrogen storage": "blueviolet",
    "battery storage 3h": "yellowgreen", "battery storage 6h": "yellowgreen",
    "electrolysis": "magenta", "turbine": "darkorange",
    "methanolisation": "cyan", "direct air capture": "coral",
    "co2 storage": "black", "methanol storage": "cadetblue",
    "electricity": "grey", "hydrogen": "grey", "co2": "grey", "methanol": "grey",
}
n_methanol.add("Carrier", list(carriers), color=list(carriers.values()))
n_methanol.add("Load", "methanol demand", bus="methanol", p_set=1e6 / 8760)
### FIN SOLUTION


## 2.4 Ajouter les renouvelables et les batteries

Les capacités éoliennes et solaires sont optimisées. Deux types de batteries sont autorisés : trois heures et six heures de stockage. Le coût annualisé combine le coût de l’onduleur et celui de la capacité énergétique.

In [ ]:
### DEBUT SOLUTION
n_methanol.add(
    "Generator", "wind", bus="electricity", carrier="wind",
    p_max_pu=ts.onwind, capital_cost=costs.at["onwind", "capital_cost"],
    p_nom_extendable=True,
)
n_methanol.add(
    "Generator", "solar", bus="electricity", carrier="solar",
    p_max_pu=ts.solar, capital_cost=costs.at["solar", "capital_cost"],
    p_nom_extendable=True,
)

for max_hours in [3, 6]:
    n_methanol.add(
        "StorageUnit", f"battery storage {max_hours}h",
        bus="electricity", carrier=f"battery storage {max_hours}h",
        max_hours=max_hours,
        capital_cost=(
            costs.at["battery inverter", "capital_cost"]
            + max_hours * costs.at["battery storage", "capital_cost"]
        ),
        efficiency_store=costs.at["battery inverter", "efficiency"],
        efficiency_dispatch=costs.at["battery inverter", "efficiency"],
        p_nom_extendable=True, cyclic_state_of_charge=True,
    )
### FIN SOLUTION


## 2.5 Ajouter la chaîne hydrogène et la capture du CO₂

L’électricité alimente l’électrolyseur, qui produit de l’hydrogène. Une turbine peut reconvertir une partie de cet hydrogène en électricité. Le CO₂ peut être capté directement dans l’air, au prix d’une consommation électrique supposée de 2 MWh par tonne de CO₂.

In [ ]:
### DEBUT SOLUTION
n_methanol.add(
    "Link", "electrolysis", bus0="electricity", bus1="hydrogen",
    carrier="electrolysis", p_nom_extendable=True,
    efficiency=costs.at["electrolysis", "efficiency"],
    capital_cost=costs.at["electrolysis", "capital_cost"],
)
n_methanol.add(
    "Link", "turbine", bus0="hydrogen", bus1="electricity",
    carrier="turbine", p_nom_extendable=True,
    efficiency=costs.at["OCGT", "efficiency"],
    capital_cost=costs.at["OCGT", "capital_cost"] / costs.at["OCGT", "efficiency"],
)

tech_h2 = "hydrogen storage tank type 1 including compressor"
n_methanol.add(
    "Store", "hydrogen storage", bus="hydrogen", carrier="hydrogen storage",
    capital_cost=costs.at[tech_h2, "capital_cost"],
    e_nom_extendable=True, e_cyclic=True,
)

electricity_input = 2  # MWh/tCO2
n_methanol.add(
    "Link", "direct air capture", bus0="electricity", bus1="co2",
    carrier="direct air capture", p_nom_extendable=True,
    efficiency=1 / electricity_input,
    capital_cost=costs.at["direct air capture", "capital_cost"] / electricity_input,
)
n_methanol.add(
    "Store", "co2 storage", bus="co2", carrier="co2 storage",
    capital_cost=costs.at["CO2 storage tank", "capital_cost"],
    e_nom_extendable=True, e_cyclic=True,
)
### FIN SOLUTION


## 2.6 Ajouter la méthanolisation et le stockage du produit

La méthanolisation consomme de l’hydrogène, de l’électricité et du CO₂ pour produire du méthanol. Les coefficients `efficiency2` et `efficiency3` sont négatifs, car ils représentent des consommations sur les bus correspondants.

Le stockage du méthanol est rattaché au bus `methanol`, afin que le produit puisse être stocké puis livré à la demande. C’est le bus cohérent avec le vecteur énergétique représenté.

In [ ]:
### DEBUT SOLUTION
eff_h2 = 1 / costs.at["methanolisation", "hydrogen-input"]

n_methanol.add(
    "Link", "methanolisation",
    bus0="hydrogen", bus1="methanol", bus2="electricity", bus3="co2",
    carrier="methanolisation", p_nom_extendable=True,
    capital_cost=costs.at["methanolisation", "capital_cost"] * eff_h2,
    efficiency=eff_h2,
    efficiency2=-costs.at["methanolisation", "electricity-input"] * eff_h2,
    efficiency3=-costs.at["methanolisation", "carbondioxide-input"] * eff_h2,
)

# Conversion du coût de stockage du liquide vers €/MWh à partir de 15,6 MJ/L.
methanol_energy_density = 15.6 * 1000 / 3600  # MWh/m³
methanol_storage_cost = (
    costs.at["General liquid hydrocarbon storage (crude)", "capital_cost"]
    / methanol_energy_density
)
n_methanol.add(
    "Store", "methanol storage", bus="methanol", carrier="methanol storage",
    capital_cost=methanol_storage_cost, e_nom_extendable=True, e_cyclic=True,
)
### FIN SOLUTION


## 2.7 Résoudre et évaluer le système

Les indicateurs permettent de comparer le coût annualisé des technologies, les capacités optimales, leur utilisation et l’énergie renouvelable écrêtée.

In [ ]:
### DEBUT SOLUTION
n_methanol.optimize()

display(n_methanol.statistics.capex().div(1e6).sort_values(ascending=False))
cout_par_mwh_methanol = (
    n_methanol.statistics.capex().sum()
    / (8760 * n_methanol.loads.p_set.sum())
)
print(f"Coût d'investissement annualisé par MWh de méthanol : {cout_par_mwh_methanol:.2f} €/MWh")

display(n_methanol.statistics.optimal_capacity())
display(n_methanol.statistics.capacity_factor() * 100)
display(n_methanol.statistics.curtailment().div(1e6))

n_base = copier_reseau(n_methanol)   # cas de base conservé pour les variantes (2.8, 2.9) et les expériences (2.10)
### FIN SOLUTION


In [ ]:
### DEBUT SOLUTION
for bus_carrier in ["electricity", "hydrogen", "co2", "methanol"]:
    n_methanol.statistics.energy_balance.plot.area(
        linewidth=0, bus_carrier=bus_carrier, figsize=(10, 3)
    )
    plt.title(f"Bilan énergétique — {bus_carrier}")
    plt.show()
### FIN SOLUTION


## 2.8 Variante A — Disponibilité de CO₂ biogénique

On ajoute une source de CO₂ biogénique disponible en quantité suffisante, à un coût marginal de 50 €/t. On compare le coût du système et les flux sur le bus CO₂ avec le cas où le carbone devait être capté directement dans l’air.

> Les variantes A et B partent toutes les deux du **cas de base** `n_base`, sauvegardé en 2.7, afin de pouvoir comparer chacune à la même référence.

In [ ]:
### DEBUT SOLUTION
n_bio = copier_reseau(n_base)
n_bio.add("Carrier", "biogenic co2", color="forestgreen")
n_bio.add(
    "Generator", "biogenic co2",
    bus="co2", carrier="biogenic co2",
    p_nom=1000, marginal_cost=50,
)
n_bio.optimize()

cout_total_biogenique = (
    n_bio.statistics.capex().sum()
    + n_bio.statistics.opex().sum()
)
print(f"Coût annualisé total avec CO₂ biogénique : {cout_total_biogenique / 1e6:.2f} M€/an")
display(n_bio.statistics.energy_balance(bus_carrier="co2"))
### FIN SOLUTION


## 2.9 Variante B — Fonctionnement moins flexible des unités PtX

On impose une puissance minimale égale à 80 % de la puissance nominale lorsque l’électrolyseur ou l’unité de méthanolisation fonctionne. Cette hypothèse réduit la flexibilité de fonctionnement et peut augmenter le besoin de stockage ou le coût total.

La variante part du cas de base `n_base` (sans le CO₂ biogénique de la variante A).

In [ ]:
### DEBUT SOLUTION
n_inflex = copier_reseau(n_base)
n_inflex.links.loc["electrolysis", "p_min_pu"] = 0.8
n_inflex.links.loc["methanolisation", "p_min_pu"] = 0.8
n_inflex.optimize()

cout_total_inflexible = (
    n_inflex.statistics.capex().sum()
    + n_inflex.statistics.opex().sum()
)
print(f"Coût annualisé total avec unités moins flexibles : {cout_total_inflexible / 1e6:.2f} M€/an")
display(n_inflex.statistics.optimal_capacity())
### FIN SOLUTION


## 2.10 Pour explorer : décomposition du coût, sensibilités et valeur du stockage

### 2.10.a D'où vient le coût du méthanol ?

On décompose le coût annualisé du cas de base (`n_base`) par technologie, en €/MWh de méthanol produit.

In [ ]:
# TODO : calculer le coût par technologie (investissement + exploitation) en €/MWh de méthanol, le tracer, et afficher le total.
### DEBUT SOLUTION
energie_methanol = 8760 * n_base.loads.p_set.sum()          # MWh de méthanol par an
couts = n_base.statistics.capex().add(n_base.statistics.opex(), fill_value=0).droplevel(0)
couts_par_mwh = (couts / energie_methanol).groupby(level=0).sum().sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(couts_par_mwh.index, couts_par_mwh.values)
ax.set_xlabel("€ par MWh de méthanol")
plt.show()
print(f"Coût total du méthanol : {couts_par_mwh.sum():.0f} €/MWh")
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

Le coût est dominé par les **renouvelables** et la **chaîne de conversion** (électrolyse, méthanolisation, captage du CO₂) : le méthanol est de l'électricité renouvelable « emballée », donc le coût de l'électricité et le rendement global de la chaîne sont déterminants. Les stockages (hydrogène, batteries, méthanol) pèsent moins mais jouent un rôle clé en permettant d'utiliser les équipements coûteux (électrolyseur, méthanolisation) avec un **facteur de charge élevé** malgré un vent et un soleil variables. Comparez avec le coût d'un carburant fossile : c'est ce qui fait de la compétitivité des carburants de synthèse une question ouverte.

### 2.10.b Sensibilités et valeur du stockage

On fait varier le coût de l'électrolyseur (×0,5 et ×2) et on retire successivement le **stockage d'hydrogène** et les **batteries**. La différence de coût avec la référence est la **valeur** de chaque stockage pour ce système.

> 💡 **Prédisez** : quel stockage a le plus de valeur ici, l'hydrogène ou les batteries ? Que se passe-t-il quand l'électrolyseur devient moins cher : sa capacité augmente-t-elle ou diminue-t-elle ?

In [ ]:
def resoudre_methanol(n_ref, facteur_electrolyse=1.0, retirer=()):
    m = copier_reseau(n_ref)
    m.links.loc["electrolysis", "capital_cost"] *= facteur_electrolyse
    for composant, nom in retirer:
        m.remove(composant, nom)
    statut, condition = m.optimize()
    assert statut == "ok", f"Optimisation non aboutie : {condition}"
    return m

In [ ]:
# TODO : résoudre les scénarios et comparer coût du méthanol (€/MWh), capacités de vent, solaire et électrolyse (GW) et
# capacité du stockage d'hydrogène (GWh).
### DEBUT SOLUTION
scenarios = {
    "référence": {},
    "électrolyseur ×0,5": {"facteur_electrolyse": 0.5},
    "électrolyseur ×2": {"facteur_electrolyse": 2.0},
    "sans stockage H2": {"retirer": [("Store", "hydrogen storage")]},
    "sans batteries": {"retirer": [("StorageUnit", "battery storage 3h"), ("StorageUnit", "battery storage 6h")]},
}
lignes = {}
for nom, options in scenarios.items():
    m = resoudre_methanol(n_base, **options)
    cap_g = m.generators.p_nom_opt
    lignes[nom] = {
        "Coût (€/MWh méthanol)": (m.statistics.capex().sum() + m.statistics.opex().sum()) / energie_methanol,
        "Vent (GW)": cap_g["wind"] / 1e3,
        "Solaire (GW)": cap_g["solar"] / 1e3,
        "Électrolyse (GW)": m.links.loc["electrolysis", "p_nom_opt"] / 1e3,
        "Stockage H2 (GWh)": m.stores.e_nom_opt.get("hydrogen storage", 0) / 1e3,
    }
tableau = pd.DataFrame(lignes).T
display(tableau.round(1))
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

- Un électrolyseur moins cher **tend à augmenter** sa capacité installée (on accepte de l'utiliser moins souvent pour profiter des pics de production renouvelable) et fait baisser le coût du méthanol ; l'inverse si son coût double. Un équipement bon marché peut être « sous-utilisé » à bon escient : c'est le même mécanisme que celui des centrales de pointe au TD long terme.
- Retirer un stockage **ne peut qu'augmenter le coût** ; l'écart mesure sa valeur. Comparez celle du stockage d'hydrogène (qui découple la production d'électrolyse de la demande de la méthanolisation) et celle des batteries (flexibilité de très court terme).
- La méthanolisation est alimentée en continu par la demande : sans stockage amont, il faut surdimensionner renouvelables et convertisseurs.

**Questions de synthèse**
1. Pourquoi un électrolyseur moins cher ne change-t-il pas seulement son propre coût, mais aussi les capacités de vent et de solaire ?
2. Quelle serait l'influence d'une unité de méthanolisation plus flexible (puissance minimale plus faible) ? Reliez-le à la variante B de 2.9.

**Questions de discussion**
- Quelles technologies dominent le coût de production du méthanol ?
- Quelle est la valeur économique du stockage d’hydrogène par rapport aux batteries ?
- Pourquoi une source de CO₂ biogénique peut-elle modifier fortement le dimensionnement du système ?
- Quel effet attendre d’une puissance minimale de fonctionnement pour l’électrolyseur et la méthanolisation ?


---

# Exercice 3 — Gestion des flux de carbone

## Objectifs

Étudier un système énergétique dans lequel le carbone est explicitement suivi entre plusieurs réservoirs :
- gaz naturel et biomasse comme ressources ;
- électricité et carburant liquide comme services énergétiques ;
- captage direct du CO₂ dans l’air (DAC) ;
- captage du carbone lors de la production d’énergie (CCS) ;
- synthèse de diesel à partir d’hydrogène et de CO₂ ;
- contrainte globale imposant des émissions nettes négatives.

Les chiffres sont fictifs et choisis pour faciliter l’analyse des arbitrages. L’objectif est surtout de comprendre la comptabilité du carbone et la façon dont une contrainte globale modifie les choix technologiques.


## 3.1 Définir les demandes et les stocks

On considère dix pas de temps. La demande d’électricité et la demande de transport sont constantes. Les stocks de diesel et d’hydrogène sont cycliques : leur niveau final doit être égal au niveau initial.

In [ ]:
### DEBUT SOLUTION
n_carbon = pypsa.Network()
n_carbon.set_snapshots(range(10))

n_carbon.add("Bus", "electricity")
n_carbon.add("Load", "electricity demand", bus="electricity", p_set=1)

n_carbon.add("Bus", "transport")
n_carbon.add("Load", "transport demand", bus="transport", p_set=1)

n_carbon.add("Bus", "diesel")
n_carbon.add("Store", "diesel", bus="diesel", e_cyclic=True, e_nom=1000)

n_carbon.add("Bus", "hydrogen")
n_carbon.add("Store", "hydrogen", bus="hydrogen", e_cyclic=True, e_nom=1000)
### FIN SOLUTION


## 3.2 Ajouter l’électrolyse

L’électrolyse transforme l’électricité en hydrogène. Son rendement est de 80 % : il faut donc 1,25 MWh d’électricité pour produire 1 MWh d’hydrogène.

In [ ]:
### DEBUT SOLUTION
n_carbon.add(
    "Link", "electrolysis",
    p_nom=2.0, efficiency=0.8,
    bus0="electricity", bus1="hydrogen",
)
### FIN SOLUTION


## 3.3 Représenter la comptabilité du CO₂

Le bus `co2 atmosphere` représente le carbone présent dans l’atmosphère sur la période modélisée. Le bus `co2 stored` représente le carbone capté et stocké, soit pour séquestration, soit comme matière première pour produire un carburant synthétique.

Dans cet exemple, le signe des flux est choisi pour rendre explicite le sens de la comptabilité : la capture retire du CO₂ de l’atmosphère, tandis que la combustion ou l’usage de carburants fossiles en ajoute.

In [ ]:
### DEBUT SOLUTION
n_carbon.add("Carrier", "co2", co2_emissions=-1)

n_carbon.add("Bus", "co2 atmosphere", carrier="co2")
n_carbon.add(
    "Store", "co2 atmosphere", bus="co2 atmosphere",
    e_nom=1000, e_min_pu=-1,
)

n_carbon.add("Bus", "co2 stored")
n_carbon.add(
    "Store", "co2 stored", bus="co2 stored",
    e_nom=1000, e_min_pu=-1,
)
### FIN SOLUTION


## 3.4 Synthétiser du diesel à partir d’hydrogène et de CO₂

Le procédé Fischer–Tropsch consomme de l’hydrogène et du CO₂ capté pour produire du diesel. Le coefficient `efficiency2` négatif représente l’entrée de CO₂ dans le procédé.

In [ ]:
### DEBUT SOLUTION
n_carbon.add(
    "Link", "Fischer-Tropsch",
    bus0="hydrogen", bus1="diesel", bus2="co2 stored",
    p_nom=4, efficiency=1, efficiency2=-1,
)
### FIN SOLUTION


## 3.5 Captage direct dans l’air et émissions du transport

Le DAC consomme de l’électricité pour déplacer du carbone du réservoir atmosphérique vers le réservoir de CO₂ capté. Le lien `diesel car` fournit le service de transport et émet du CO₂ dans l’atmosphère.

In [ ]:
### DEBUT SOLUTION
n_carbon.add(
    "Link", "DAC",
    bus0="electricity", bus1="co2 stored", bus2="co2 atmosphere",
    efficiency=1, efficiency2=-1, p_nom=5,
)
n_carbon.add(
    "Link", "diesel car",
    bus0="diesel", bus1="transport", bus2="co2 atmosphere",
    efficiency=1, efficiency2=1, p_nom=2,
)
### FIN SOLUTION


## 3.6 Ajouter le gaz naturel, avec et sans captage

Le gaz est représenté par un stock limité dont le coût marginal vaut 20 €/MWh thermique. Deux centrales sont disponibles :
- une turbine à gaz classique, qui émet le carbone dans l’atmosphère ;
- une centrale avec captage et stockage du carbone (CCS), dont le captage est imparfait.

In [ ]:
### DEBUT SOLUTION
n_carbon.add("Bus", "gas")
n_carbon.add(
    "Store", "gas", bus="gas",
    e_initial=50, e_nom=50, marginal_cost=20,
)

n_carbon.add(
    "Link", "OCGT",
    bus0="gas", bus1="electricity", bus2="co2 atmosphere",
    p_nom_extendable=True, efficiency=0.5, efficiency2=1,
)
n_carbon.add(
    "Link", "OCGT+CCS",
    bus0="gas", bus1="electricity",
    bus2="co2 stored", bus3="co2 atmosphere",
    p_nom_extendable=True, efficiency=0.4,
    efficiency2=0.9, efficiency3=0.1,
)
### FIN SOLUTION


## 3.7 Ajouter la biomasse, avec et sans captage

La biomasse est une ressource limitée et coûteuse. Sa combustion sans captage est considérée comme neutre dans la comptabilité simplifiée, car le carbone émis est supposé provenir de la biomasse récemment absorbée. Avec captage, le CO₂ biogénique peut être stocké durablement : cela peut contribuer à atteindre des émissions nettes négatives.

In [ ]:
### DEBUT SOLUTION
n_carbon.add("Bus", "biomass")
n_carbon.add(
    "Store", "biomass", bus="biomass",
    marginal_cost=30, e_nom=55, e_initial=55,
)

n_carbon.add(
    "Link", "biomass",
    bus0="biomass", bus1="electricity",
    p_nom_extendable=True, capital_cost=1, efficiency=0.5,
)
n_carbon.add(
    "Link", "biomass+CCS",
    bus0="biomass", bus1="electricity",
    bus2="co2 stored", bus3="co2 atmosphere",
    p_nom_extendable=True, capital_cost=1,
    efficiency=0.4, efficiency2=1, efficiency3=-1,
)
### FIN SOLUTION


## 3.8 Imposer une cible d’émissions nettes négatives

La contrainte globale fixe une cible de −50 sur la période modélisée. Une valeur négative signifie que le système doit retirer davantage de carbone de l’atmosphère qu’il n’en émet, selon la comptabilité du modèle.

In [ ]:
### DEBUT SOLUTION
n_carbon.add(
    "GlobalConstraint", "co2_limit",
    sense="<=", constant=-50,
)
n_carbon.optimize()
### FIN SOLUTION


## 3.9 Analyser les stocks et les flux

Les stocks indiquent comment les ressources et le carbone évoluent au fil des pas de temps. Les flux de puissance des liens permettent d’identifier les technologies mobilisées pour satisfaire les demandes et respecter la cible climatique.

In [ ]:
### DEBUT SOLUTION
n_carbon.stores_t.e.plot(figsize=(10, 4), title="Évolution des stocks")
plt.ylabel("Unité énergétique ou quantité de carbone")
plt.show()

n_carbon.links_t.p0.plot(figsize=(10, 4), title="Flux à l'entrée des procédés")
plt.ylabel("Puissance / flux du modèle")
plt.show()

carbone_suivi = n_carbon.stores_t.e[
    ["co2 stored", "co2 atmosphere", "gas", "diesel"]
].sum(axis=1)
display(carbone_suivi)
### FIN SOLUTION


## 3.10 Pour explorer : prix implicite du carbone, cible et ressource en biomasse

Le modèle a une **contrainte globale** sur les émissions nettes. Sa variable duale est le **prix implicite du CO₂** : le coût marginal de la décarbonation dans ce système, c'est-à-dire ce que coûterait une unité de CO₂ supplémentaire à retirer de l'atmosphère. La fonction suivante (fournie) résout une variante du réseau, en changeant la cible d'émissions ou la disponibilité de biomasse, et renvoie le résultat (ou `None` si le problème est infaisable).

In [ ]:
def resoudre_carbone(n_ref, cible=None, facteur_biomasse=None):
    """Résout une variante de n_ref. Renvoie le réseau, ou None si le problème est infaisable."""
    m = copier_reseau(n_ref)
    if cible is not None:
        m.global_constraints.loc["co2_limit", "constant"] = cible
    if facteur_biomasse is not None:
        m.stores.loc["biomass", ["e_nom", "e_initial"]] = 55 * facteur_biomasse
    statut, condition = m.optimize()
    return m if statut == "ok" else None

### 3.10.a Vérifier la comptabilité du carbone

> 💡 **Prédisez** : quelle valeur doit avoir le stock `co2 atmosphere` à la dernière période si la contrainte est active ? Que vaut le prix implicite du CO₂ dans le cas à −50 ?

In [ ]:
# TODO : afficher le niveau final du stock atmosphérique, le niveau de carbone stocké, le coût optimal et le prix implicite du CO2
# (attention au signe de mu).
### DEBUT SOLUTION
print("Stock atmosphérique final :", round(n_carbon.stores_t.e["co2 atmosphere"].iloc[-1], 2))
print("Carbone stocké final       :", round(n_carbon.stores_t.e["co2 stored"].iloc[-1], 2))
print("Coût optimal               :", round(n_carbon.objective, 1))
print("Prix implicite du CO2      :", round(-n_carbon.global_constraints.mu["co2_limit"], 2), "€/unité de CO2")
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

Le stock atmosphérique final est **égal à la cible (−50)** : la contrainte est active, le modèle va exactement jusqu'à la limite. Le prix implicite du CO₂ est strictement positif : c'est le coût marginal d'une unité de CO₂ supplémentaire retirée (PyPSA renvoie `mu` négatif pour une contrainte « ≤ », d'où le signe moins). Une contrainte non active aurait un prix implicite nul.

### 3.10.b Quel est le coût d'une cible plus ambitieuse ?

On balaie la cible d'émissions nettes de +100 (pratiquement pas de contrainte) à −60, et on relève le coût optimal, le prix implicite du CO₂ et les capacités des quatre centrales.

> 💡 **Prédisez** : le coût augmente-t-il linéairement quand la cible devient plus négative ? Existe-t-il une cible **inatteignable** ?

In [ ]:
# TODO : balayer les cibles [100, 0, -10, -20, -30, -40, -50, -60] ; pour chaque cible faisable, relever coût, prix implicite et
# capacités des centrales. Marquer les cibles infaisables par NaN. Tracer coût et prix implicite.
### DEBUT SOLUTION
lignes = {}
for cible in [100, 0, -10, -20, -30, -40, -50, -60]:
    m = resoudre_carbone(n_carbon, cible=cible)
    if m is None:
        lignes[cible] = {"Coût": np.nan, "Prix implicite du CO2": np.nan}
        continue
    cap = m.links.p_nom_opt
    lignes[cible] = {
        "Coût": m.objective,
        "Prix implicite du CO2": -m.global_constraints.mu["co2_limit"],
        **{nom: cap[nom] for nom in ["OCGT", "OCGT+CCS", "biomass", "biomass+CCS"]},
    }
tableau_cible = pd.DataFrame(lignes).T.rename_axis("Cible d'émissions nettes")
display(tableau_cible.round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
tableau_cible["Coût"].plot(ax=axes[0], marker="o", title="Coût optimal")
tableau_cible["Prix implicite du CO2"].plot(ax=axes[1], marker="o", drawstyle="steps-mid", title="Prix implicite du CO2")
for ax in axes:
    ax.set_xlabel("Cible d'émissions nettes")
plt.tight_layout()
plt.show()
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

*Valeurs vérifiées sur ce modèle (PyPSA 1.4, HiGHS).*
- Le coût **n'augmente pas linéairement** : le prix implicite est constant (≈ 9,2) pour les cibles de +100 à environ −40, puis **saute** (≈ 66,7 pour −50) : c'est la **courbe de coût marginal d'abattement**, croissante par paliers. Au début on utilise des options peu coûteuses (substitution du gaz par du gaz avec CCS et de la biomasse avec CCS) ; ensuite, la biomasse étant épuisée, il faut compenser avec des solutions plus chères.
- Le mix change : l'OCGT sans captage disparaît dès que la cible est contraignante, le CCS le remplace, et la part de la biomasse avec CCS augmente avec l'ambition (c'est la seule source d'**émissions négatives** abordable).
- Pour une cible d'environ −60, le problème est **infaisable** : le système ne peut pas physiquement retirer davantage de carbone, car les ressources (biomasse), la capacité du DAC (5) et la demande énergétique plafonnent les émissions négatives possibles. Un problème infaisable est une information : il indique la borne du système.

### 3.10.c La biomasse, ressource limitante

On garde la cible de −50 et on fait varier la quantité de biomasse disponible (×0,5, ×1, ×1,5, ×2).

In [ ]:
# TODO : pour chaque facteur de biomasse, relever la faisabilité, le coût et le prix implicite du CO2.
### DEBUT SOLUTION
lignes = {}
for facteur in [0.5, 1.0, 1.5, 2.0]:
    m = resoudre_carbone(n_carbon, facteur_biomasse=facteur)
    if m is None:
        lignes[facteur] = {"Faisable": False, "Coût": np.nan, "Prix implicite du CO2": np.nan}
    else:
        lignes[facteur] = {"Faisable": True, "Coût": m.objective, "Prix implicite du CO2": -m.global_constraints.mu["co2_limit"]}
display(pd.DataFrame(lignes).T.rename_axis("Facteur de biomasse").round(1))
### FIN SOLUTION

**Éléments de réponse** *(cellule de correction)*

Avec moitié moins de biomasse, la cible de −50 est **infaisable** ; avec la biomasse de référence, elle est atteinte à un prix implicite élevé (≈ 67) ; avec **deux fois plus** de biomasse, le coût baisse d'environ 12 % et le prix implicite chute (≈ 21). La biomasse avec CCS est donc **la** ressource rare qui limite la cible : sa disponibilité détermine à la fois la faisabilité et le coût marginal de la décarbonation. Plus généralement, le **prix implicite d'une contrainte** (ici le CO₂) est un indicateur de rareté : il se lit comme la valeur d'une unité supplémentaire de la ressource limitante.

**Questions de synthèse**
1. Pourquoi le prix implicite du CO₂ est-il constant sur une plage de cibles, puis augmente-t-il brutalement ?
2. Que faudrait-il modifier dans le modèle (quelle capacité, quelle ressource) pour rendre la cible de −60 faisable ?

**Questions de discussion**
- Pourquoi la production de diesel synthétique ne constitue-t-elle pas automatiquement une élimination permanente du CO₂ ?
- Quelle différence entre captage du carbone et séquestration durable ?
- Pourquoi le CCS appliqué à la biomasse peut-il contribuer aux émissions nettes négatives ?
- Quels arbitrages la contrainte `co2_limit` impose-t-elle entre gaz, biomasse, DAC et CCS ?


---

# Synthèse du TD

| Exercice | Systèmes couplés | Compétence principale | Phénomène mis en évidence |
|---|---|---|---|
| Modèle à un nœud | Électricité, chaleur, hydrogène, mobilité | Construire un réseau multi-vecteurs | Prix couplés (chaleur = électricité / COP), valeur des options de flexibilité |
| Méthanol isolé | Renouvelables, batteries, hydrogène, CO₂, méthanol | Optimiser un système PtX autonome | Décomposition du coût, valeur du stockage, sous-utilisation volontaire des équipements |
| Gestion du carbone | Électricité, carburants, biomasse, DAC, CCS | Suivre les flux de carbone et imposer une cible | Prix implicite du CO₂, coût d'abattement croissant, faisabilité |

## Pour aller plus loin
1. Comparer la production de méthanol en Argentine et en Namibie (variante en commentaire en 2.2).
2. Dans l'exercice 1, faire varier le coût de la pompe à chaleur et trouver le coût à partir duquel elle disparaît du mix.
3. Dans l'exercice 3, ajouter une pénalité sur la biomasse (coût marginal plus élevé) et suivre l'évolution du prix implicite du CO₂.
4. Raffiner la cible d'émissions entre −50 et −60 pour trouver précisément le point où le problème devient infaisable.